In [1]:
from db_connection import setup_sakila, save_result_csv

engine = setup_sakila(displaylimit=None)

displaylimit: Value None will be treated as 0 (no limit)


サブクエリ（Subquery）とは、`1つのSQL文の中に含まれる別のSELECT文`である。

メインクエリの条件やデータとして使用される。

使用する位置によって、次の3種類に分けられる。

- スカラサブクエリ
- インラインビュー
- サブクエリ

```sql
SELECT ..., (SELECT ...)
FROM ...;
```

```sql
SELECT ...
FROM ... (SELECT ...);
```

```sql
SELECT ...
FROM ...
WHERE カラム名 = (SELECT ...);
```

## 1. スカラサブクエリ（Scalar Subquery）

- `1行1列`の値のみを返すサブクエリである。

- 主に`SELECT句`や`WHERE句`で使用される。

#### 例. 映画の平均レンタル料金と一緒に表示する

In [3]:
%%sql

SELECT title,
       rental_rate,
       (SELECT AVG(rental_rate) FROM film) AS avg_rate
FROM film

LIMIT 5;

title,rental_rate,avg_rate
ACADEMY DINOSAUR,0.99,2.980000
ACE GOLDFINGER,4.99,2.980000
ADAPTATION HOLES,2.99,2.980000
AFFAIR PREJUDICE,2.99,2.980000
AFRICAN EGG,2.99,2.980000


## 2. インラインビュー（Inline View）

- `FROM句`で使用するサブクエリである。

- サブクエリの実行結果を、一時的なテーブル（View）のように使用する。

#### 例. 映画の料金が4.99以上のデータを取得する

In [5]:
%%sql

SELECT *
FROM (
    SELECT title, rental_rate
    FROM film
) AS f
WHERE rental_rate >= 4.99

LIMIT 5;

title,rental_rate
ACE GOLDFINGER,4.99
AIRPLANE SIERRA,4.99
AIRPORT POLLOCK,4.99
ALADDIN CALENDAR,4.99
ALI FOREVER,4.99


## 3. WHERE句のサブクエリ

`WHERE句`の条件値として使用するサブクエリである。

`1行`または`複数行`の結果を返すことができる。

#### 例. 映画の平均上映時間より長い映画を取得する

In [7]:
%%sql

SELECT title, length
FROM film
WHERE length > (
    SELECT AVG(length)
    FROM film
)

LIMIT 5;

title,length
AFFAIR PREJUDICE,117
AFRICAN EGG,130
AGENT TRUMAN,169
ALAMO VIDEOTAPE,126
ALASKA PHANTOM,136


#### - WHERE句のサブクエリが1列・複数行を返す場合

サブクエリが`1列・複数行`を返す場合は、`IN`や`NOT IN`などを使用する。

次の例では、指定した2つの映画と同じレーティングを持たない映画を取得する。

In [ ]:
%%sql

SELECT *
FROM film
WHERE rating NOT IN (
    SELECT rating
    FROM film
    WHERE title = 'ADAPTATION HOLES'
       OR title = 'ACE GOLDFINGER' -- G, NC17
)

LIMIT 5;

film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,special_features,last_update
1,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist who must Battle a Teacher in The Canadian Rockies,2006,1,None,6,0.99,86,20.99,PG,"Deleted Scenes,Behind the Scenes",2006-02-15 05:03:42
6,AGENT TRUMAN,A Intrepid Panorama of a Robot And a Boy who must Escape a Sumo Wrestler in Ancient China,2006,1,None,3,2.99,169,17.99,PG,Deleted Scenes,2006-02-15 05:03:42
7,AIRPLANE SIERRA,A Touching Saga of a Hunter And a Butler who must Discover a Butler in A Jet Boat,2006,1,None,6,4.99,62,28.99,PG-13,"Trailers,Deleted Scenes",2006-02-15 05:03:42
8,AIRPORT POLLOCK,A Epic Tale of a Moose And a Girl who must Confront a Monkey in Ancient India,2006,1,None,6,4.99,54,15.99,R,Trailers,2006-02-15 05:03:42
9,ALABAMA DEVIL,A Thoughtful Panorama of a Database Administrator And a Mad Scientist who must Outgun a Mad Scientist in A Jet Boat,2006,1,None,3,2.99,114,21.99,PG-13,"Trailers,Deleted Scenes",2006-02-15 05:03:42


#### - WHERE句のサブクエリが1行・複数列を返す場合

サブクエリが`1行・複数列`を返す場合は、複数のカラムを1つの組として比較する。

次の例では、`ACE GOLDFINGER`と同じレーティング・レンタル料金を持つ映画を取得する。

In [ ]:
%%sql

SELECT *
FROM film
WHERE (rating, rental_rate) = (
    SELECT rating, rental_rate
    FROM film
    WHERE title = 'ACE GOLDFINGER' -- G, 4.99
)

LIMIT 5;

film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,special_features,last_update
2,ACE GOLDFINGER,A Astounding Epistle of a Database Administrator And a Explorer who must Find a Car in Ancient China,2006,1,None,3,4.99,48,12.99,G,"Trailers,Deleted Scenes",2006-02-15 05:03:42
46,AUTUMN CROW,A Beautiful Tale of a Dentist And a Mad Cow who must Battle a Moose in The Sahara Desert,2006,1,None,3,4.99,108,13.99,G,"Trailers,Commentaries,Deleted Scenes,Behind the Scenes",2006-02-15 05:03:42
61,BEAUTY GREASE,A Fast-Paced Display of a Composer And a Moose who must Sink a Robot in An Abandoned Mine Shaft,2006,1,None,5,4.99,175,28.99,G,"Trailers,Commentaries",2006-02-15 05:03:42
75,BIRD INDEPENDENCE,A Thrilling Documentary of a Car And a Student who must Sink a Hunter in The Canadian Rockies,2006,1,None,6,4.99,163,14.99,G,"Commentaries,Behind the Scenes",2006-02-15 05:03:42
77,BIRDS PERDITION,A Boring Story of a Womanizer And a Pioneer who must Face a Dog in California,2006,1,None,5,4.99,61,15.99,G,"Trailers,Behind the Scenes",2006-02-15 05:03:42


#### # WHERE句のサブクエリが複数行・複数列を返す場合

サブクエリが`複数行・複数列`を返す場合は、複数のカラムを1つの組として`IN`で比較する。

次の例では、指定した2つの映画と同じレーティング・レンタル料金を持つ映画を取得する。

In [ ]:
%%sql

SELECT *
FROM film
WHERE (rating, rental_rate) IN (
    SELECT rating, rental_rate
    FROM film
    WHERE title = 'ACE GOLDFINGER' -- (G, 4.99)
       OR title = 'ADAPTATION HOLES' -- (NC-17, 2.99)
)

LIMIT 5;

film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,special_features,last_update
2,ACE GOLDFINGER,A Astounding Epistle of a Database Administrator And a Explorer who must Find a Car in Ancient China,2006,1,None,3,4.99,48,12.99,G,"Trailers,Deleted Scenes",2006-02-15 05:03:42
3,ADAPTATION HOLES,A Astounding Reflection of a Lumberjack And a Car who must Sink a Lumberjack in A Baloon Factory,2006,1,None,7,2.99,50,18.99,NC-17,"Trailers,Deleted Scenes",2006-02-15 05:03:42
15,ALIEN CENTER,A Brilliant Drama of a Cat And a Mad Scientist who must Battle a Feminist in A MySQL Convention,2006,1,None,5,2.99,46,10.99,NC-17,"Trailers,Commentaries,Behind the Scenes",2006-02-15 05:03:42
16,ALLEY EVOLUTION,A Fast-Paced Drama of a Robot And a Composer who must Battle a Astronaut in New Orleans,2006,1,None,6,2.99,180,23.99,NC-17,"Trailers,Commentaries",2006-02-15 05:03:42
29,ANTITRUST TOMATOES,A Fateful Yarn of a Womanizer And a Feminist who must Succumb a Database Administrator in Ancient India,2006,1,None,5,2.99,168,11.99,NC-17,"Trailers,Commentaries,Deleted Scenes",2006-02-15 05:03:42
